In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

import pandas as pd

import dianne
dianne.setNotebookWidth(100)

In [ ]:
# Load the STQ data and prepare the patches for annotation and training the classifier
dataPath = '../../data/'
samples  = ['JDC-WP-012-w-STQ']

classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)

fname = 'img.data.ctranspath-1.h5ad'
ads, imgs, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne.loadDataAndPreparePatches(samples, dataPath, fname=fname, L=None, ts=56, mpp=0.25, N=8)

In [ ]:
# Load the demo Xenium data and cell annotations
dataSubPathsXeniumSlim = ['xenium-slim-JDC-WP-012-w']
for xpath in dataSubPathsXeniumSlim:
    dianne.downloadFromZenodo(dataPath + xpath, 'https://zenodo.org/api/records/15777586/files-archive')
xeniumBundlePaths = {sample: dataPath + xpath for sample, xpath in zip(samples, dataSubPathsXeniumSlim)}
xeniumMatrixPaths = {sample: dataPath + xpath + '/matrix.csv' for sample, xpath in zip(samples, dataSubPathsXeniumSlim)}
annotations = {sample: pd.read_csv(xeniumBundlePaths[sample] + '/annotated_fine.csv', index_col=0)['group'] for sample in samples}
uannotations = set(a for sample in samples for a in annotations[sample].unique())
annotationsPalette = {a: dianne.Set123(i) for i, a in enumerate(uannotations)}

In [ ]:
# Display representative patches that can be used to start the annotation process
startParams = {}
dianne.jumpStart(patchCoordinates, patchesCDFs, imgs, startParams, ncols=6, nrows=3, ads=ads, pyramidscale=2,
                 L=1 if L is None else L, sh=int(ts/2)/mpp, figsize=(3, 3), seed=1, maxN=10**3)

In [ ]:
clf, plog, bp = {}, [], {}
try: bp.update({startParams['selected_patch']: 'positive'})
except: pass

In [ ]:
clf, bp, plog, startParams = dianne.loadClassifier(classifierPaths, 'classifier-IPMN')

In [ ]:
# Run the annotation loop with active learning and probabilistic patch selection
dianne.runAnnotation(patchCoordinates, patchesCDFs, imgs, bp, clf, plog, ads=ads, qs=qs, minN=1, pyramidscale=2,
                    figsize=(5, 5), augFunc=dianne.PCMA, alpha=0.5, R=1, cmapColors=['lightcoral', 'blue'], # (5, 5)
                    seed=0, randomness=0.5, L=1 if L is None else L, sh=int(ts/2)/mpp,
                    xeniumBundlePaths=xeniumBundlePaths, xeniumMatrixPaths=xeniumMatrixPaths, numColumns=1,
                    transcriptsAlpha=0.05, transcriptsColor='lime', transcriptsSize=2., loadTranscripts=False,
                    selectedGenes=None, minCount=1, loadCells=True, loadCellBoundaries=True,
                    showAnnotations=True, annotations=annotations, annotationsPalette=annotationsPalette)

In [ ]:
dianne.saveClassifier(clf, classifierPaths, 'classifier-IPMN', dataPath, samples, patchesCDFs, L, ts, mpp, N, fname, qs, startParams, plog, bp)

In [ ]:
# To run inference on the entire slides
infSample = samples[0]
x, y, p = dianne.inferProb(ads[infSample], clf['clf'], qs, tsize=ts/mpp, R=2, erode=False)
dianne.showProbImg(x, y, p, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample)